## Creating STAC Items for SHYFEM Icechunk Stores

This notebook creates [STAC](https://stacspec.org/) **Items** — one per Icechunk store — for
[SHYFEM](https://www.cmcc.it/models/shyfem) unstructured-grid ocean model output produced by the
[GlobalCoast](https://globalcoastalservices.eu/) project.

We have two sites:
- **Taranto** — Gulf of Taranto, Mediterranean (SHYFEM-MPI, CMCC)
- **Antsiranana** — Antsiranana Bay, Madagascar (SHYFEM, ISMAR-CNR)

Both are stored as Icechunk repositories on an EGI MinIO object store.

**Why Items (not a Collection)?**  
A STAC Item represents a single granule with its own spatial footprint and time range.
Keeping each model run as a separate Item means we can build a
[stac-geoparquet](https://github.com/stac-utils/stac-geoparquet) index and run fast
spatial/temporal queries with [rustac](https://github.com/gadomski/rustac) —
which is the payoff shown at the end of this notebook.

**Steps**
1. Define store configurations in a list
2. Loop over stores: open each Icechunk store, derive bbox/geometry from the UGRID mesh, build a `pystac.Item`, and add the Icechunk asset
3. Validate and save each Item to JSON
4. Round-trip: reload each Item from JSON and re-open the store
5. Build a local STAC Catalog and write it to disk
6. Write a GeoParquet index with `rustac` and run spatial/temporal queries
7. *(Optional)* Upload the catalog tree to S3 with `fsspec`

## 1. Imports & credentials

In [ ]:
import json
import os
import warnings

import icechunk
import pandas as pd
import pystac
import rustac
import shapely.geometry
import xarray as xr
import xstac
import xugrid as xu
from dotenv import load_dotenv

warnings.filterwarnings(
    "ignore",
    message="Numcodecs codecs are not in the Zarr version 3 specification*",
    category=UserWarning,
)

load_dotenv(os.path.expanduser("~/dotenv/protocoast.env"))

ENDPOINT_URL = os.environ["ENDPOINT_URL"]
BUCKET = "protocoast-data"

## 2. Helper: open a SHYFEM Icechunk store

The stores live on an EGI MinIO instance that requires path-style addressing
(`force_path_style=True`) and authenticated access.  
We open the `main` branch and read the current `snapshot_id` so the STAC Item
pins an exact, reproducible version of the store.

In [ ]:
def open_shyfem_store(prefix):
    """Open a SHYFEM Icechunk store on Tubitak RustFS bucket and return (repo, session, uds, ds)."""
    storage = icechunk.s3_storage(
        endpoint_url=ENDPOINT_URL,
        bucket=BUCKET,
        prefix=prefix,
        force_path_style=True,
        from_env=True,
    )
    repo = icechunk.Repository.open(
        storage,
        authorize_virtual_chunk_access={f"s3://{BUCKET}/": None},
    )
    session = repo.readonly_session(branch="main")
    ds = xr.open_zarr(session.store, consolidated=False)
    uds = xu.UgridDataset(ds)
    return repo, session, uds, ds


def mesh_bbox_and_geometry(uds):
    """Return (bbox, geometry) derived from the xugrid mesh bounding polygon."""
    grid = uds.ugrid.grids[0]
    xmin, ymin, xmax, ymax = grid.bounds
    bbox = [float(xmin), float(ymin), float(xmax), float(ymax)]
    geometry = shapely.geometry.mapping(grid.bounding_polygon())
    return bbox, geometry


def derive_temporal_extent(ds):
    """Derive [start, end] ISO strings from CF forecast coordinates.

    Both SHYFEM stores use a forecast encoding: a ``time`` coordinate
    (the reference time, scalar or 1-D) plus a ``step`` coordinate
    (timedelta forecast periods). The valid-time extent is therefore:
        start = time[0] + step[0]
        end   = time[-1] + step[-1]
    """
    t = pd.Timestamp(ds["time"].values.ravel()[0])
    t_end = pd.Timestamp(ds["time"].values.ravel()[-1])
    s = ds["step"].values
    start = t     + pd.Timedelta(s[0])
    end   = t_end + pd.Timedelta(s[-1])
    fmt = "%Y-%m-%dT%H:%M:%SZ"
    return start.strftime(fmt), end.strftime(fmt)


def model_from_source(ds):
    """Extract model name from the CF 'source' global attribute.

    E.g. 'Model data produced by SHYFEM-MPI at CMCC' -> 'SHYFEM-MPI'
    """
    source = ds.attrs.get("source", "")
    if "produced by" in source:
        return source.split("produced by")[1].split(" at ")[0].strip()
    return source


def build_cube_dimensions(ds, start_datetime, end_datetime):
    """Build cube:dimensions for a SHYFEM UGRID dataset.

    xstac cannot auto-detect these because the dataset uses forecast-style
    time encoding (timedelta 'step' + reference 'time') and an unstructured
    UGRID spatial grid rather than regular lat/lon axes.

    Dimension types follow the datacube extension v2.2.0 schema:
    - temporal: type='temporal', no axis allowed
    - vertical: type='spatial', axis='z'
    - unstructured node: type must NOT be 'spatial' or 'geometry' per
      additional_dimension schema, so we use type='other'
    """
    level = ds["level"]
    return {
        "step": {
            "type":        "temporal",
            "extent":      [start_datetime, end_datetime],
            "description": "Forecast lead time (valid time = reference time + step)",
        },
        "level": {
            "type":   "spatial",
            "axis":   "z",
            "extent": [float(level.values[0]), float(level.values[-1])],
            "unit":   level.attrs.get("units", "m"),
            "description": level.attrs.get("standard_name", "depth"),
        },
        "node": {
            "type":        "other",
            "extent":      [0, int(ds.sizes["node"]) - 1],
            "description": "UGRID unstructured mesh node index; lon/lat stored as 1-D node coordinates",
        },
    }

## 2. Store configurations

Each entry defines the Icechunk prefix and the metadata that cannot be derived
from the dataset itself (title, description, temporal extent, model, institution).

In [ ]:
STORES = [
    {
        "name":        "taranto",
        "prefix":      "icechunk/taranto-icechunk-tubitak-v1",
        "title":       "SHYFEM Taranto (Gulf of Taranto, Mediterranean)",
        "description": (
            "SHYFEM-MPI unstructured grid ocean model output for the Gulf of Taranto, "
            "produced by CMCC as part of the GlobalCoast project."
        ),
    },
    {
        "name":        "antsiranana",
        "prefix":      "icechunk/antsiranana-icechunk-tubitak",
        "title":       "SHYFEM Antsiranana Bay, Madagascar",
        "description": (
            "SHYFEM unstructured grid ocean model output for Antsiranana Bay, Madagascar, "
            "produced by ISMAR-CNR as part of the GlobalCoast project."
        ),
    },
]

## 3. Open stores and build STAC Items

For each store we:
1. Open the Icechunk repository and read the current snapshot id
2. Wrap the dataset as a `UgridDataset` and derive bbox + boundary polygon from the mesh
3. Build a `pystac.Item` and populate it with `xstac`
4. Add the Icechunk asset with storage/zarr/virtual-assets extension fields

In [ ]:
STORAGE_SCHEME_KEY = "egi-minio"

STAC_EXTENSIONS = [
    "https://stac-extensions.github.io/datacube/v2.2.0/schema.json",
    "https://stac-extensions.github.io/zarr/v1.1.0/schema.json",
    "https://stac-extensions.github.io/storage/v2.0.0/schema.json",
    "https://stac-extensions.github.io/virtual-assets/v1.0.0/schema.json",
    "https://stac-extensions.github.io/version/v1.2.0/schema.json",
]

items = []

for cfg in STORES:
    print(f"\n--- {cfg['name']} ---")
    repo, session, uds, ds = open_shyfem_store(cfg["prefix"])
    snapshot_id = session.snapshot_id
    print(f"snapshot_id: {snapshot_id}")

    bbox, geometry = mesh_bbox_and_geometry(uds)
    print(f"bbox: {bbox}")

    start_datetime, end_datetime = derive_temporal_extent(ds)
    print(f"temporal extent: {start_datetime} / {end_datetime}")

    # Pull metadata directly from dataset global attributes
    institution = ds.attrs["institution"]
    model       = model_from_source(ds)
    conventions = ds.attrs.get("Conventions", "CF-1.4")
    print(f"model: {model}  institution: {institution}")

    template = pystac.Item(
        id=f"{cfg['name']}-icechunk@{snapshot_id}",
        geometry=geometry,
        bbox=bbox,
        datetime=None,
        properties={
            "title":          cfg["title"],
            "description":    cfg["description"],
            "start_datetime": start_datetime,
            "end_datetime":   end_datetime,
            "model":          model,
            "institution":    institution,
            "conventions":    conventions,
            "variables":      ["temperature", "salinity", "u_velocity", "v_velocity", "water_level"],
        },
        stac_extensions=STAC_EXTENSIONS,
    )

    # temporal_dimension=False: forecast-style encoding (timedelta 'step' + reference
    # 'time') can't be converted to datetime by xstac. x/y=False: no regular lat/lon
    # axes on a UGRID mesh. We build cube:dimensions manually below.
    item = xstac.xarray_to_stac(
        ds, template,
        temporal_dimension=False,
        x_dimension=False,
        y_dimension=False,
        reference_system=4326,
        validate=False,
    )

    # Replace the empty cube:dimensions xstac leaves behind with meaningful entries
    item.properties["cube:dimensions"] = build_cube_dimensions(ds, start_datetime, end_datetime)

    # storage:schemes goes in item.properties per storage extension v2.0.0 schema
    item.properties["storage:schemes"] = {
        STORAGE_SCHEME_KEY: {
            "type":             "custom-s3",
            "platform":         f"{ENDPOINT_URL}/{{bucket}}",
            "bucket":           BUCKET,
            "endpoint_url":     ENDPOINT_URL,
            "force_path_style": True,
            "anonymous":        False,
        }
    }

    item.add_asset(
        f"{cfg['name']}-icechunk@{snapshot_id}",
        pystac.Asset(
            href=f"s3://{BUCKET}/{cfg['prefix']}",
            title=f"SHYFEM {cfg['title'].split('(')[0].strip()} — Icechunk Store",
            media_type="application/vnd.zarr+icechunk",
            roles=["data", "references", "virtual", "latest-version"],
            extra_fields={
                "zarr:consolidated":    False,
                "zarr:zarr_format":     3,
                "icechunk:snapshot_id": snapshot_id,
                "storage:refs":         [STORAGE_SCHEME_KEY],
            },
        ),
    )

    items.append(item)
    print(f"built item: {item.id}")

items

## 4. Validate and save

In [ ]:
for item in items:
    item.validate()
    fname = f"{item.id.split('@')[0]}_stac_item.json"
    with open(fname, "w") as f:
        json.dump(item.to_dict(), f, indent=2)
    print(f"Saved → {fname}")

## 4b. Round-trip: reload each Item from JSON and re-open the store

Reconstruct the Icechunk store from the saved Item JSON using only the fields
defined in the STAC extensions — no hard-coded credentials or paths.

In [ ]:
for item in items:
    fname = f"{item.id.split('@')[0]}_stac_item.json"
    loaded = pystac.Item.from_file(fname)

    ic_asset = next(
        a for a in loaded.assets.values()
        if a.media_type == "application/vnd.zarr+icechunk"
    )
    snap_id = ic_asset.extra_fields["icechunk:snapshot_id"]
    ref     = ic_asset.extra_fields["storage:refs"][0]
    scheme  = loaded.properties["storage:schemes"][ref]
    prefix  = ic_asset.href.split(f"{scheme['bucket']}/")[1]

    storage = icechunk.s3_storage(
        endpoint_url=scheme["endpoint_url"],
        bucket=scheme["bucket"],
        prefix=prefix,
        force_path_style=scheme["force_path_style"],
        from_env=True,
    )
    repo    = icechunk.Repository.open(
        storage,
        authorize_virtual_chunk_access={f"s3://{scheme['bucket']}/": None},
    )
    session = repo.readonly_session(snapshot_id=snap_id)
    ds      = xr.open_zarr(session.store, consolidated=False)
    print(f"{loaded.id}: {ds}")
    print()

## 5. Build a local STAC Catalog

Wrap all items in a `pystac.Catalog`, normalize hrefs to a local directory,
and write the full tree to disk with `catalog.save()`.

In [ ]:
import shutil

CATALOG_DIR = os.path.join(os.getcwd(), "shyfem-stac")
if os.path.exists(CATALOG_DIR):
    shutil.rmtree(CATALOG_DIR)

catalog = pystac.Catalog(
    id="globalcoast-shyfem",
    description="GlobalCoast SHYFEM icechunk stores — unstructured ocean model output",
    title="GlobalCoast SHYFEM Catalog",
)
for item in items:
    catalog.add_item(item)

catalog.normalize_hrefs(CATALOG_DIR)
catalog.save(catalog_type=pystac.CatalogType.SELF_CONTAINED)
print(f"Catalog written to {CATALOG_DIR}/")

## 6. Write a GeoParquet index and query with rustac

`rustac.geoparquet_writer` serialises the items to a GeoParquet file, after which
`rustac.search_sync` can run fast spatial and temporal queries without touching the
individual JSON files.

In [ ]:
PARQUET_PATH = os.path.join(CATALOG_DIR, "catalog.parquet")
items_dicts = [item.to_dict() for item in items]

async with rustac.geoparquet_writer(items_dicts[:1], PARQUET_PATH) as w:
    if len(items_dicts) > 1:
        await w.write(items_dicts[1:])

print(f"Saved → {PARQUET_PATH}")

In [ ]:
# Temporal query: which stores cover March 2021?
results = rustac.search_sync(
    PARQUET_PATH,
    datetime="2021-03-31T00:00:00Z/2021-04-06T00:00:00Z",
)
print([item["id"] for item in results])

In [ ]:
# Spatial query: which stores intersect a point in the Gulf of Taranto?
results = rustac.search_sync(
    PARQUET_PATH,
    intersects={"type": "Point", "coordinates": [17.5, 39.8]},
)
print([item["id"] for item in results])

In [ ]:
# Open the first result directly from the geoparquet metadata
hit = results[0]

# Locate the Icechunk asset (match on icechunk:snapshot_id being present)
ic_asset = next(a for a in hit["assets"].values() if "icechunk:snapshot_id" in a)

snap_id = ic_asset["icechunk:snapshot_id"]
ref     = ic_asset["storage:refs"][0]
scheme  = hit["properties"]["storage:schemes"][ref]
prefix  = ic_asset["href"].split(f"{scheme['bucket']}/")[1]

print(f"item:        {hit['id']}")
print(f"snapshot_id: {snap_id}")
print(f"prefix:      {prefix}")
print(f"endpoint:    {scheme['endpoint_url']}")

storage = icechunk.s3_storage(
    endpoint_url=scheme["endpoint_url"],
    bucket=scheme["bucket"],
    prefix=prefix,
    force_path_style=scheme["force_path_style"],
    from_env=True,
)
repo    = icechunk.Repository.open(
    storage,
    authorize_virtual_chunk_access={f"s3://{scheme['bucket']}/": None},
)
session = repo.readonly_session(snapshot_id=snap_id)
ds      = xr.open_zarr(session.store, consolidated=False)
ds

## 7. Upload to S3 *(optional)*

Upload the local catalog tree (JSON files + GeoParquet) to `s3://protocoast-data/stac/`
using `fsspec`.  This makes the catalog discoverable at a public URL for production use.

> **Requires** valid `AWS_ACCESS_KEY_ID` / `AWS_SECRET_ACCESS_KEY` in your environment.

In [ ]:
import fsspec

fs = fsspec.filesystem(
    "s3",
    endpoint_url=ENDPOINT_URL,
    key=os.environ["AWS_ACCESS_KEY_ID"],
    secret=os.environ["AWS_SECRET_ACCESS_KEY"],
)

fs.put("shyfem-stac/", f"{BUCKET}/stac/", recursive=True)
print(f"Uploaded shyfem-stac/ → s3://{BUCKET}/stac/")